# LLM signal：构造、设定与导出

**这个 notebook 做什么**

1. 定义 baseline 回归（Attention channel）与需要哪些交互项
2. 从新闻级预测构造两种事件级 LLM signal
3. 描述统计与相关性
4. 导出 `.dta`，回归在 Stata 里用 `reghdfe` 跑

**上游**：事件级大表 `build/pead_panel.parquet`（构造见 `README.md` 与 `数据准备.ipynb`），
已含 CAR、SUE、10 个控制变量、NRANK / ATT。

**只读数据源**：
`/project/dachxiu/yifei/news/experiment/US/ARTICLE/RidgeProximal/QUESTION_CHOICE_pred_1d_.../pred_YYYY.pkl`

---

# 1. Baseline 回归设定

## 1.1 变量

| 记号 | 变量 | 含义 |
|---|---|---|
| $CAR^{ANN}$ | `car_ann_o2o` | 公告窗口 $[d,\ d+1]$ 的累计异常收益，open-to-open |
| $CAR^{DRIFT}$ | `car_drift_o2o` | 漂移窗口 $[d+2,\ d+61]$ 的累计异常收益 |
| $SUE$ | `sue_rank` | 盈余意外 $(e-F)/P$ 的当季十分位，缩放到 $[0,1]$ |
| $ATT$ | `att` | $11-NRANK$，NRANK 为同日公告数的季度内十分位。取值 1–10，**越大注意力越充裕** |
| $LLM$ | `llm_first` / `llm_avg` | 事件窗口内新闻的 LLM 预测，两种口径见 §2 |
| $X_k$ | 10 个 | SIZE、BM、LNANALYST、LAG、LAG²、LAG³、IO、EVOL、EPERSIST、TURN |

收益口径用 **O2O**：`pred` 的训练目标是 open-to-open 收益，与之同源。

固定效应：年 + 月 + 星期几 + FF10 行业（follow HLT 2009 Table III）。
标准误按**公告日**聚类。

## 1.2 需要哪些交互项

核心系数是 $ATT\times LLM$：**注意力充裕时，LLM signal 对 CAR 的作用是否不同**。
围绕它，三个变量 $\{SUE,\ ATT,\ LLM\}$ 之间有三个两两交互 —— $SUE\times ATT$、
$SUE\times LLM$、$ATT\times LLM$。哪些必须放，取决于它们之间的相关性：
若某个两两交互被遗漏、而它与核心交互相关，核心系数就有偏。相关性由 §3.1 的 cell 实测给出。

设定分三档，核心系数在三档下都报，看它稳不稳：

**(1) 最简**

$$CAR^{w}_{i,d}=\alpha+b_1 SUE+b_2 ATT+b_3 LLM
+\underbrace{b_4\,(ATT\times LLM)}_{\text{核心}}
+\sum_{k=1}^{10}c_k X_k+\eta_t+\psi_j+\varepsilon$$

**(2) 加 SUE 的两个交互**（推荐作主表）

$$CAR^{w}_{i,d}=\ \cdots\ +b_5(SUE\times ATT)+b_6(SUE\times LLM)+\ \cdots$$

三个变量的两两交互全齐。

**(3) HLT 完整**

$$CAR^{w}_{i,d}=\ \cdots\ +\sum_k d_k(X_k\times SUE)+\sum_k f_k(X_k\times LLM)+\ \cdots$$

HLT 2009 eq. (4) 把控制变量与盈余意外全部交互，理由是这些变量本身会改变市场对信息的
**敏感度**。同理 $X_k\times LLM$ 保护 $b_4$、$X_k\times SUE$ 保护 $b_5$。

交互项在 Stata 里用因子记号 `c.x##c.z` 现算，不预先生成。

---

# 2. LLM signal 的构造

## 2.1 数据源与结构

`pred_YYYY.pkl`，2004-01 至 2026-03，逐年一个 DataFrame，**新闻级**：

| 列 | 含义 |
|---|---|
| `timestamp` | 新闻时间（ET，毫秒精度）—— 判断归属与排序都用它。**时区标注不统一**：2004–2018 无时区（本身即 ET 墙上时间，小时分布与后段一致），2019–2026 带 `America/New_York`，读入时统一成 ET 无时区 |
| `PERMNO` | 公司 |
| `DATE` | 新闻对应的可交易日（盘后新闻已推到次日）。本 notebook 不用它划窗口 |
| `pred` | 该条新闻对 $open_{DATE}\to open_{DATE+1}$ 收益的预测 |
| `labels` | 同一天的实现收益（firm-day 级，同一天所有新闻共享） |

一天可以有多条新闻：约三成 firm-day 有多行，最多一天 48 条。

## 2.2 归属规则：timestamp 落在事件窗口内

一条新闻属于某个事件，当且仅当它的 **`timestamp` 落在该事件的 $[d,\ d+1]$ 窗口内**，
其中 $d$ 是公告后的第一个交易日（大表的 `td0`）。

窗口按**连续时间段**取：从交易日 $d$ 当天 00:00（ET）到交易日 $d+1$ 当天 23:59:59（ET）。

- 盘前、盘中、盘后的新闻都算，只要时间戳落在这两个交易日的日历跨度内
- $d$ 是周五、$d+1$ 是周一时，**周末的新闻也算进来** —— 按日历日逐日匹配会把它们漏掉

`timestamp` 同时用于排序，定位窗口内最早的那一条。

## 2.3 两种口径

窗口内公司 $i$ 的全部新闻记为 $\{n_1,\dots,n_N\}$，按 `timestamp` 升序。

**口径 A —— 首条新闻**

$$LLM^{first}_{i,d}=pred(n_1)$$

时间戳最早那条新闻的预测值。含义是"事件的第一反应"，不被后续新闻稀释。

**口径 B —— 全窗口按条平均**

$$LLM^{avg}_{i,d}=\frac{1}{N}\sum_{j=1}^{N} pred(n_j)$$

分母是**新闻总条数** $N$，不是天数 —— 某天新闻多，那天的权重就大。

两种口径都不做跨日累乘或累加。

In [1]:
import glob
import os
import pickle

import numpy as np
import pandas as pd

DATA, BUILD = "data", "build"
SRC = ("/project/dachxiu/yifei/news/experiment/US/ARTICLE/RidgeProximal/"
       "QUESTION_CHOICE_pred_1d_is4cv3_cossim_0.8_O2O_RET_future_1d_O2O_RET"
       "_not_rank_normed_rolling_move_trading_days_expectation_only")

WIN = (0, 1)          # 事件窗口 [d, d+1]，与 CAR[0,1] 对齐

# 交易日历：与大表 td0_idx 同一套
cal = pd.concat([pd.read_parquet(f, columns=["dlycaldt"])["dlycaldt"].drop_duplicates()
                 for f in sorted(glob.glob(f"{DATA}/crsp_daily_*.parquet"))])
cal = pd.to_datetime(cal).drop_duplicates().sort_values().reset_index(drop=True)
print(f"交易日历 {len(cal):,} 天: {cal.iloc[0].date()} ~ {cal.iloc[-1].date()}")

panel = pd.read_parquet(f"{BUILD}/pead_panel.parquet")
ev = panel[["eid", "permno", "anndats", "td0", "td0_idx"]].dropna(subset=["td0_idx"]).copy()
ev["permno"] = ev["permno"].astype("int32")
ev["td0_idx"] = ev["td0_idx"].astype("int32")

# 窗口的起止日历日：交易日 d 与交易日 d+1 的日期
ev["win_start"] = cal.reindex(ev["td0_idx"] + WIN[0]).to_numpy()
ev["win_end"] = cal.reindex(np.minimum(ev["td0_idx"] + WIN[1], len(cal) - 1)).to_numpy()
print(f"事件 {len(ev):,}")
print(ev[["eid", "permno", "td0", "win_start", "win_end"]].head().to_string(index=False))

交易日历 7,673 天: 1996-01-02 ~ 2026-06-30


事件 517,955
 eid  permno        td0  win_start    win_end
   0   75554 1996-01-02 1996-01-02 1996-01-03
   1   16791 1996-01-02 1996-01-02 1996-01-03
   2   79713 1996-01-02 1996-01-02 1996-01-03
   3   64697 1996-01-02 1996-01-02 1996-01-03
   4   13100 1996-01-02 1996-01-02 1996-01-03


In [2]:
# 把每个事件的窗口展开成"公司 × 日历日"，用来和新闻的时间戳日期做等值连接。
# 窗口跨越周末时（d 周五、d+1 周一）中间的日历日也包含在内，周末新闻不会被漏掉。
span = (ev["win_end"] - ev["win_start"]).dt.days
print(f"窗口跨越的日历天数分布: {span.value_counts().sort_index().to_dict()}")

keys = []
for k in range(int(span.max()) + 1):
    sub = ev.loc[span >= k, ["eid", "permno", "win_start"]].copy()
    sub["cdate"] = sub["win_start"] + pd.Timedelta(days=k)
    keys.append(sub[["eid", "permno", "cdate"]])
keys = pd.concat(keys, ignore_index=True)
print(f"展开后的 (事件, 公司, 日历日) 键: {len(keys):,}")

窗口跨越的日历天数分布: {1: 473825, 2: 501, 3: 39649, 4: 3905, 5: 63, 7: 12}
展开后的 (事件, 公司, 日历日) 键: 1,127,748


In [3]:
# 逐年读新闻，按 timestamp 的 ET 日历日与窗口做匹配
parts = []
for f in sorted(glob.glob(f"{SRC}/pred_*.pkl")):
    if f.endswith("_r2s.pkl"):
        continue
    with open(f, "rb") as fh:
        d = pickle.load(fh)
    d = d.reset_index()[["timestamp", "PERMNO", "pred"]]
    d["permno"] = pd.to_numeric(d["PERMNO"], errors="coerce").astype("int32")
    # 时区不统一：2004-2018 无时区（本身就是 ET 墙上时间），2019-2026 带 America/New_York。
    # 统一成 ET 无时区，否则 concat 后会退化成 object、排序报错。
    ts = pd.to_datetime(d["timestamp"])
    if getattr(ts.dtype, "tz", None) is not None:
        ts = ts.dt.tz_convert("America/New_York").dt.tz_localize(None)
    d["timestamp"] = ts
    d["cdate"] = ts.dt.normalize()
    m = d.merge(keys, on=["permno", "cdate"], how="inner")
    if len(m):
        parts.append(m[["eid", "timestamp", "pred"]])
    print(f"  {os.path.basename(f)}: {len(d):,} 条 → 落在事件窗口内 {len(m):,}", flush=True)
    del d, m

news = pd.concat(parts, ignore_index=True)
del parts
print(f"\n窗口内新闻合计 {len(news):,} 条，涉及 {news['eid'].nunique():,} 个事件")

  pred_2004.pkl: 100,113 条 → 落在事件窗口内 15,987


  pred_2005.pkl: 120,007 条 → 落在事件窗口内 18,019


  pred_2006.pkl: 120,256 条 → 落在事件窗口内 18,028


  pred_2007.pkl: 142,610 条 → 落在事件窗口内 20,792


  pred_2008.pkl: 157,444 条 → 落在事件窗口内 22,997


...

  pred_2023.pkl: 152,661 条 → 落在事件窗口内 31,155


  pred_2024.pkl: 155,388 条 → 落在事件窗口内 31,104


  pred_2025.pkl: 160,351 条 → 落在事件窗口内 35,959


  pred_2026.pkl: 23,033 条 → 落在事件窗口内 6,453



窗口内新闻合计 577,193 条，涉及 188,979 个事件


In [4]:
# timestamp 排序后取最早一条；均值按条数
news = news.sort_values(["eid", "timestamp"], kind="mergesort")

sig = news.groupby("eid").agg(
    llm_first=("pred", "first"),          # 口径 A：窗口内时间戳最早的一条
    llm_avg=("pred", "mean"),             # 口径 B：窗口内所有新闻按条平均
    llm_n=("pred", "size"),               # 窗口内新闻条数
    llm_ts_first=("timestamp", "first"),
    llm_ts_last=("timestamp", "last"),
).reset_index()

sig = ev[["eid", "permno", "anndats", "td0", "td0_idx"]].merge(sig, on="eid", how="left")
sig["llm_n"] = sig["llm_n"].fillna(0).astype("int32")

sig.to_parquet(f"{BUILD}/llm_event_signal.parquet", index=False)
print(f"→ {BUILD}/llm_event_signal.parquet  {len(sig):,} 行")
print(f"有信号的事件: {int((sig['llm_n'] > 0).sum()):,} ({(sig['llm_n'] > 0).mean():.1%})\n")
print(sig[sig["llm_n"] > 0].head(8).to_string(index=False))

→ build/llm_event_signal.parquet  517,955 行
有信号的事件: 188,979 (36.5%)

   eid  permno    anndats        td0  td0_idx  llm_first   llm_avg  llm_n            llm_ts_first             llm_ts_last
179357   74500 2004-01-06 2004-01-06     2017  -0.000052 -0.000052      1 2004-01-06 16:03:53.680 2004-01-06 16:03:53.680
179362   34948 2004-01-07 2004-01-07     2018   0.000993  0.001385      2 2004-01-07 16:01:46.123 2004-01-07 16:36:48.995
179369   75976 2004-01-07 2004-01-07     2018   0.001053  0.001053      1 2004-01-07 08:50:44.856 2004-01-07 08:50:44.856
179372   79307 2004-01-07 2004-01-07     2018   0.000516  0.000516      1 2004-01-07 09:25:01.251 2004-01-07 09:25:01.251
179373   79866 2004-01-07 2004-01-07     2018   0.000875  0.000389      2 2004-01-07 08:02:46.343 2004-01-07 15:12:21.069
179375   85272 2004-01-07 2004-01-07     2018   0.001006  0.001006      1 2004-01-07 07:02:16.235 2004-01-07 07:02:16.235
179377   87114 2004-01-07 2004-01-07     2018   0.001130  0.001233      2 200

---

# 3. 描述统计与相关性

In [5]:
s = sig[sig["llm_n"] > 0]
print(f"有信号的事件 {len(s):,} | 窗口内新闻条数：中位 {s['llm_n'].median():.0f}、"
      f"均值 {s['llm_n'].mean():.2f}、最大 {s['llm_n'].max():,}")
print(f"只有 1 条新闻的占 {(s['llm_n'] == 1).mean():.1%}（这时两种口径完全相同）\n")

print("两种口径的分布")
print(s[["llm_first", "llm_avg"]].describe(percentiles=[.01, .25, .5, .75, .99]).round(6).to_string())
print(f"\n两者相关系数: {s['llm_first'].corr(s['llm_avg']):.4f}")

print("\n按窗口内新闻条数分组")
g = s.assign(bin=pd.cut(s["llm_n"], [0, 1, 2, 5, 10, 10 ** 6],
                        labels=["1 条", "2 条", "3-5 条", "6-10 条", ">10 条"]))
print(g.groupby("bin", observed=True).apply(
    lambda x: pd.Series({"事件数": len(x),
                         "first 均值": x["llm_first"].mean(),
                         "avg 均值": x["llm_avg"].mean(),
                         "两者相关": x["llm_first"].corr(x["llm_avg"])}),
    include_groups=False).round(6).to_string())

有信号的事件 188,979 | 窗口内新闻条数：中位 2、均值 3.05、最大 69
只有 1 条新闻的占 12.2%（这时两种口径完全相同）

两种口径的分布
           llm_first        llm_avg
count  188979.000000  188979.000000
mean        0.000810       0.000587
std         0.001525       0.001423
min        -0.008991      -0.007407
1%         -0.003413      -0.003136
25%         0.000000      -0.000226
50%         0.000856       0.000668
75%         0.001644       0.001425
99%         0.004811       0.004220
max         0.009345       0.008166



两者相关系数: 0.7972

按窗口内新闻条数分组


            事件数  first 均值    avg 均值      两者相关
bin                                          
1 条     23079.0  0.000701  0.000701  1.000000
2 条     72939.0  0.000854  0.000614  0.820332
3-5 条   75675.0  0.000835  0.000557  0.750244
6-10 条  15224.0  0.000694  0.000467  0.641551
>10 条    2062.0  0.000448  0.000305  0.554832


In [6]:
# 与 SUE、ATT、CAR 的相关性
FF10 = [
    ("NoDur", [(100, 999), (2000, 2399), (2700, 2749), (2770, 2799), (3100, 3199), (3940, 3989)]),
    ("Durbl", [(2500, 2519), (2590, 2599), (3630, 3659), (3710, 3711), (3714, 3714), (3716, 3716),
               (3750, 3751), (3792, 3792), (3900, 3939), (3990, 3999)]),
    ("Manuf", [(2520, 2589), (2600, 2699), (2750, 2769), (3000, 3099), (3200, 3569), (3580, 3629),
               (3700, 3709), (3712, 3713), (3715, 3715), (3717, 3749), (3752, 3791), (3793, 3799),
               (3830, 3839), (3860, 3899)]),
    ("Enrgy", [(1200, 1399), (2900, 2999)]),
    ("HiTec", [(3570, 3579), (3660, 3692), (3694, 3699), (3810, 3829), (7370, 7379), (7391, 7391),
               (8730, 8734)]),
    ("Telcm", [(4800, 4899)]),
    ("Shops", [(5000, 5999), (7200, 7299), (7600, 7699)]),
    ("Hlth",  [(2830, 2839), (3693, 3693), (3840, 3859), (8000, 8099)]),
    ("Utils", [(4900, 4949)]),
]


def ff10(sic):
    x = pd.to_numeric(sic, errors="coerce")
    out = pd.Series("Other", index=x.index, dtype=object)
    done = pd.Series(False, index=x.index)
    for name, rngs in FF10:
        hit = pd.Series(False, index=x.index)
        for lo, hi in rngs:
            hit |= x.between(lo, hi)
        out[hit & ~done] = name
        done |= hit
    out[x.isna()] = np.nan
    return out


CTRL = ["size_dec", "bm_dec", "lnanalyst", "lag", "lag2", "lag3", "io", "evol", "epersist", "turn"]
CARS = ["car_ann_o2o", "car_drift_o2o"]

df = panel.merge(sig.drop(columns=["permno", "anndats", "td0", "td0_idx"]), on="eid", how="left")
df["ff10"] = ff10(df["siccd"])
df = df[df["is_latest_pends_on_day"] & ~df["flag_lag_bad"]]
df = df.dropna(subset=["sue", "sue_dec", "att", "ff10"] + CTRL + CARS)
df["sue_rank"] = (df["sue_dec"] - 1) / 9.0
df["date_id"] = pd.factorize(df["anndats"])[0]
reg = df[df["llm_n"] > 0].copy()

print(f"baseline 样本 {len(df):,} | 其中有 LLM signal {len(reg):,} ({len(reg) / len(df):.1%})")
print(f"有信号样本的年份 {reg['anndats'].dt.year.min()}-{reg['anndats'].dt.year.max()}\n")
print("相关系数（有信号样本）")
print(reg[["sue_rank", "llm_first", "llm_avg", "att"] + CARS].corr().round(4).to_string())

baseline 样本 298,385 | 其中有 LLM signal 138,523 (46.4%)
有信号样本的年份 2004-2025

相关系数（有信号样本）
               sue_rank  llm_first  llm_avg     att  car_ann_o2o  car_drift_o2o
sue_rank         1.0000     0.1635   0.2305 -0.0029       0.3293         0.0410
llm_first        0.1635     1.0000   0.7825  0.0168       0.1751         0.0200
llm_avg          0.2305     0.7825   1.0000  0.0336       0.2495         0.0197
att             -0.0029     0.0168   0.0336  1.0000       0.0097        -0.0049
car_ann_o2o      0.3293     0.1751   0.2495  0.0097       1.0000         0.0226
car_drift_o2o    0.0410     0.0200   0.0197 -0.0049       0.0226         1.0000


## 3.1 三个变量的相关性：决定要放哪些交互项

§1.2 说的三档设定，取舍依据就是下面这张表。

In [7]:
# 决定交互项用的相关性矩阵：三个变量两两之间
KEY = ["sue_rank", "att", "llm_first", "llm_avg"]
print("Pearson")
print(reg[KEY].corr().round(4).to_string())
print("\nSpearman")
print(reg[KEY].corr(method="spearman").round(4).to_string())

print("\n判断依据")
for c in ["llm_first", "llm_avg"]:
    r_sue = reg["sue_rank"].corr(reg[c])
    r_att = reg["att"].corr(reg[c])
    print(f"  {c}: 与 SUE {r_sue:+.3f}，与 ATT {r_att:+.3f}"
          f"  →  SUE x LLM {'需要保留' if abs(r_sue) > 0.05 else '影响很小'}")
print(f"  SUE 与 ATT {reg['sue_rank'].corr(reg['att']):+.3f}"
      f"  →  两者近乎正交，SUE x ATT 对核心系数的干扰有限，但它是 HLT 的基准结果，仍然报出")

Pearson
           sue_rank     att  llm_first  llm_avg
sue_rank     1.0000 -0.0029     0.1635   0.2305
att         -0.0029  1.0000     0.0168   0.0336
llm_first    0.1635  0.0168     1.0000   0.7825
llm_avg      0.2305  0.0336     0.7825   1.0000

Spearman
           sue_rank     att  llm_first  llm_avg
sue_rank     1.0000 -0.0040     0.1683   0.2342
att         -0.0040  1.0000     0.0169   0.0337
llm_first    0.1683  0.0169     1.0000   0.7491
llm_avg      0.2342  0.0337     0.7491   1.0000

判断依据
  llm_first: 与 SUE +0.163，与 ATT +0.017  →  SUE x LLM 需要保留
  llm_avg: 与 SUE +0.231，与 ATT +0.034  →  SUE x LLM 需要保留
  SUE 与 ATT -0.003  →  两者近乎正交，SUE x ATT 对核心系数的干扰有限，但它是 HLT 的基准结果，仍然报出


In [8]:
# LLM signal 在 SUE 十分位上的形态
t = reg.groupby("sue_dec").agg(N=("eid", "size"))
for c in ["llm_first", "llm_avg"]:
    t[c] = (reg.groupby("sue_dec")[c].mean() * 1e4).round(2)
t.index = [f"D{int(i)}" for i in t.index]
t.columns = ["N", "llm_first (x1e4)", "llm_avg (x1e4)"]
print("LLM signal 按 SUE 十分位（数值乘以 1e4）")
print(t.to_string())

LLM signal 按 SUE 十分位（数值乘以 1e4）
         N  llm_first (x1e4)  llm_avg (x1e4)
D1    9395              1.40           -2.62
D2   12636              3.14           -0.47
D3   14843              5.87            2.47
D4   15283              8.49            5.96
D5   16266             10.10            8.55
D6   15850             11.00            9.53
D7   15222             11.20            9.90
D8   14599             11.10            9.73
D9   13494             10.63            9.12
D10  10935              8.84            6.46


---

---

# 4. 导出给 Stata

导出**原始变量**，交互项在 Stata 里用因子记号 `##` 现算 —— 这样 `margins`、`contrast`
等后估计命令可以正常使用，`##` 也会自动带上主效应。

```stata
use build/analysis_llm.dta, clear

* (1) 最简
reghdfe car_drift_o2o sue_rank c.att##c.llm_first ///
    size_dec bm_dec lnanalyst lag lag2 lag3 io evol epersist turn ///
    , absorb(year month dow ff10) vce(cluster date_id)

* (2) 加 SUE 的两个交互（主表）
reghdfe car_drift_o2o c.att##c.llm_first c.sue_rank##c.att c.sue_rank##c.llm_first ///
    size_dec bm_dec lnanalyst lag lag2 lag3 io evol epersist turn ///
    , absorb(year month dow ff10) vce(cluster date_id)

* (3) HLT 完整：控制变量分别与 SUE、LLM 交互
reghdfe car_drift_o2o c.att##c.llm_first c.sue_rank##c.att c.sue_rank##c.llm_first ///
    c.(size_dec bm_dec lnanalyst lag lag2 lag3 io evol epersist turn)##c.sue_rank ///
    c.(size_dec bm_dec lnanalyst lag lag2 lag3 io evol epersist turn)##c.llm_first ///
    , absorb(year month dow ff10) vce(cluster date_id)
```

把 `llm_first` 换成 `llm_avg` 即另一种口径，`car_drift_o2o` 换成 `car_ann_o2o` 即另一个窗口。

## 4.1 `analysis_llm.dta` 列字典

### 键与固定效应

| 列名 | 英文全称 | 含义 |
|---|---|---|
| `eid` | Event ID | 事件唯一编号，可回溯到 `pead_panel.parquet` |
| `permno` | CRSP Permanent Number | 公司永久标识 |
| `anndats` | Announcement Date | 盈余公告日 |
| `year` `month` `dow` | Year / Month / Day-of-Week | 日历固定效应 |
| `ff10` | Fama-French 10 Industry | 行业固定效应，由 SIC 映射 |
| `date_id` | Announcement-date ID | 公告日的整数编号，**聚类维度** |

### 被解释变量

| 列名 | 英文全称 | 含义 |
|---|---|---|
| `car_ann_o2o` | CAR, Announcement window, Open-to-Open | 窗口 $[d,\ d+1]$ 的累计异常收益。**主口径**，与 `pred` 的训练目标同为 open-to-open |
| `car_drift_o2o` | CAR, Drift window, Open-to-Open | 窗口 $[d+2,\ d+61]$ |
| `car_ann_c2c` `car_drift_c2c` | CAR, Close-to-Close | 同两个窗口的收盘价口径，作对照 |

### 盈余意外

| 列名 | 英文全称 | 含义 |
|---|---|---|
| `sue` | Standardized Unexpected Earnings | 原始值 $(e-F)/P$，HLT 2009 记作 FE (Forecast Error) |
| `sue_dec` | SUE decile | 按公告所在**日历季度**排的十分位，1 = 最负、10 = 最正 |
| **`sue_rank`** | SUE rank | $(sue\_dec-1)/9$，取值 $[0,1]$，**进回归的就是它**。系数直接读作 D10 相对 D1 的差异 |

### 注意力

| 列名 | 英文全称 | 含义 |
|---|---|---|
| `n_ann_day` | Number of announcements that day | 公告当日全市场的季报公告家数 |
| `nrank` | Number-of-announcements decile | 上一列在**日历季度内**的十分位，1 = 当日公告最少、10 = 最多（最分心） |
| **`att`** | Attention | $11-NRANK$，取值 1–10，**数值越大注意力越充裕** |

### LLM signal

| 列名 | 英文全称 | 含义 |
|---|---|---|
| `llm_n` | Number of news items | 事件窗口内的新闻条数 |
| **`llm_first`** | LLM signal, first news item | 窗口内**时间戳最早**那条新闻的 `pred` |
| **`llm_avg`** | LLM signal, average across news | 窗口内所有新闻 `pred` 的**按条平均** |
| `llm_ts_first` `llm_ts_last` | Timestamp of first / last news | 窗口内首末条新闻的时间戳，核对用 |

两个 signal 都是原始值，与 CAR 同在收益量纲上 —— 系数读作
"LLM 信号每高 1 个单位，CAR 变化多少"。

### 控制变量（10，HLT 2009 §III.A）

| 列名 | 英文全称 | 含义 |
|---|---|---|
| `size_dec` | Firm Size decile | formation 年 6 月末市值，NYSE 断点分十位 |
| `bm_dec` | Book-to-Market decile | $BE/ME$，NYSE 断点分十位 |
| `lnanalyst` | Log(1 + Number of Analysts) | $\log(1+\#\{\text{公告前 365 天出过预测的分析师}\})$ |
| `lag` `lag2` `lag3` | Reporting Lag and its powers | 公告日 − 财季结束日（天），及平方、三次方 |
| `io` | Institutional Ownership | 公告前最近一期 13F 持股比例 |
| `evol` | Earnings Volatility | 过去 16 财季 $\Delta_4 EPS$ 的标准差 |
| `epersist` | Earnings Persistence | 过去 16 财季**季度 EPS 水平值**的一阶自相关 |
| `turn` | Share Turnover | 过去 12 个月的月均换手率 |

### 样本

只保留同时满足以下条件的事件：同日多季报取最新一期（`is_latest_pends_on_day`）、
报告滞后正常（非 `flag_lag_bad`）、10 个控制变量与 CAR 齐全、**事件窗口内至少有一条新闻**。

In [9]:
KEEP = (["eid", "permno", "anndats", "year", "month", "dow", "ff10", "date_id",
         "car_ann_o2o", "car_drift_o2o", "car_ann_c2c", "car_drift_c2c",
         "sue", "sue_dec", "sue_rank",
         "n_ann_day", "nrank", "att",
         "llm_first", "llm_avg", "llm_n", "llm_ts_first", "llm_ts_last"] + CTRL)
out = reg[[c for c in KEEP if c in reg.columns]].copy()
out["ff10"] = out["ff10"].astype("category")

out.to_stata(f"{BUILD}/analysis_llm.dta", write_index=False, version=118)
out.to_parquet(f"{BUILD}/analysis_llm.parquet", index=False)
print(f"→ {BUILD}/analysis_llm.dta   {len(out):,} 行 × {out.shape[1]} 列")
print(f"→ {BUILD}/analysis_llm.parquet")
print(f"年份 {out['anndats'].dt.year.min()}-{out['anndats'].dt.year.max()} | "
      f"{out['permno'].nunique():,} 家公司 | {out['date_id'].nunique():,} 个公告日\n")
print(out.dtypes.to_frame("dtype").to_string())

→ build/analysis_llm.dta   138,523 行 × 33 列
→ build/analysis_llm.parquet
年份 2004-2025 | 4,015 家公司 | 5,069 个公告日

                        dtype
eid                     int64
permno                  int64
anndats        datetime64[ns]
year                    int32
month                   int32
dow                     int32
ff10                 category
date_id                 int64
car_ann_o2o           float64
car_drift_o2o         float64
car_ann_c2c           float64
car_drift_c2c         float64
sue                   Float64
sue_dec               float64
sue_rank              float64
n_ann_day             float64
nrank                 float64
att                   float64
llm_first             float64
llm_avg               float64
llm_n                   int32
llm_ts_first   datetime64[ns]
llm_ts_last    datetime64[ns]
size_dec              float64
bm_dec                float64
lnanalyst             float64
lag                     int64
lag2                    int64
lag3              

# 5. 回归结果

样本 138,523 个事件、4,015 家公司、2004–2025 年。估计用 Stata `reghdfe`，
吸收年、月、星期几、FF10 行业四组固定效应，标准误按公告日聚类。

三档设定各一张表，每张 4 列 —— 窗口作外层、$LLM$ 口径作内层：

| | (1) | (2) | (3) | (4) |
|:---|:---|:---|:---|:---|
| 窗口 | ANN | ANN | DRIFT | DRIFT |
| 口径 | first | average | first | average |

相邻两列比换口径系数变不变，跨中线比换窗口系数变不变。
正表报核心系数与主效应，10 个控制变量与它们的交互以 Yes / No 标示；
全部系数见 §6 附录。

## 5.1 设定 (1)：最简

$$
CAR^{w}_{i,d}=\alpha
+b_1\,SUE_{i,d}+b_2\,ATT_{i,d}+b_3\,LLM_{i,d}
+\underbrace{b_4\,(ATT_{i,d}\times LLM_{i,d})}_{\text{核心}}
+\sum_{k=1}^{10}c_k X_{k,i,d}
+\eta_{year}+\eta_{month}+\eta_{dow}+\psi_{ff10}+\varepsilon_{i,d}
$$

**表 1　最简设定**

| | (1) | (2) | (3) | (4) |
|:---|---:|---:|---:|---:|
| 被解释变量 | $CAR^{ANN}$ | $CAR^{ANN}$ | $CAR^{DRIFT}$ | $CAR^{DRIFT}$ |
| 窗口 | $[d,\,d{+}1]$ | $[d,\,d{+}1]$ | $[d{+}2,\,d{+}61]$ | $[d{+}2,\,d{+}61]$ |
| $LLM$ 口径 | first | average | first | average |
| $SUE$ | 0.0774\*\*\* | 0.0692\*\*\* | 0.0269\*\*\* | 0.0262\*\*\* |
| | (0.0008) | (0.0008) | (0.0025) | (0.0026) |
| $ATT$ | 0.0004\*\*\* | 0.0002 | 0.0003 | 0.0003 |
| | (0.0001) | (0.0001) | (0.0004) | (0.0004) |
| $LLM$ | 9.3520\*\*\* | 14.0910\*\*\* | 4.4314\*\*\* | 4.9897\*\*\* |
| | (0.4576) | (0.5458) | (1.1769) | (1.4748) |
| **$ATT\times LLM$** | −0.2456\*\*\* | −0.0668 | −0.3182\* | −0.4151\* |
| | (0.0663) | (0.0776) | (0.1700) | (0.2178) |
| | | | | |
| 控制变量（10） | Yes | Yes | Yes | Yes |
| 控制变量 $\times\,SUE$ | No | No | No | No |
| 控制变量 $\times\,LLM$ | No | No | No | No |
| 年 / 月 / 星期 / 行业 FE | Yes | Yes | Yes | Yes |
| 标准误聚类 | 公告日 | 公告日 | 公告日 | 公告日 |
| 观测数 | 138,523 | 138,523 | 138,523 | 138,523 |
| Within $R^2$ | 0.1287 | 0.1523 | 0.0024 | 0.0024 |

*注：* 系数下方括号内为按公告日聚类的稳健标准误（5,069 个 cluster）。\* $p<0.10$，\*\* $p<0.05$，\*\*\* $p<0.01$。$SUE$ 为盈余意外的当季十分位缩放到 $[0,1]$；$ATT=11-NRANK$，取值 1–10；$LLM$ 为事件窗口 $[d,\,d{+}1]$ 内新闻的 LLM 预测，first 取时间戳最早一条、average 取全部新闻按条平均。控制变量 10 个：SIZE、BM、LNANALYST、LAG、LAG²、LAG³、IO、EVOL、EPERSIST、TURN。收益口径为 open-to-open。样本 2004–2025 年，4,015 家公司。

**表中读到的**

- $b_4$ 四列全为负。DRIFT 两列 $p$ 值 0.061 与 0.057；ANN 两列中 first 为 −0.2456
  （$p<0.01$）、average 为 −0.0668（$p=0.39$）。
- $b_3$ 四列全为正且 $p<0.01$。ANN 列 average 的 14.0910 是 first 9.3520 的 1.51 倍；
  DRIFT 列两者 4.9897 与 4.4314，相差 12.6%。
- $b_1$ 四列全为正且 $p<0.01$，ANN 约 0.07、DRIFT 约 0.027。
- $b_2$ 仅 ANN 列 first 达到 $p<0.01$，其余三列 $p>0.10$。
- Within $R^2$：ANN 列 0.1287 与 0.1523，DRIFT 列均为 0.0024。

## 5.2 设定 (2)：三个两两交互全齐

$$
CAR^{w}_{i,d}=\ \cdots
+\underbrace{b_4\,(ATT\times LLM)}_{\text{核心}}
+b_5\,(SUE\times ATT)+b_6\,(SUE\times LLM)
+\sum_{k=1}^{10}c_k X_k+\ \cdots
$$

**表 2　三个两两交互全齐**

| | (1) | (2) | (3) | (4) |
|:---|---:|---:|---:|---:|
| 被解释变量 | $CAR^{ANN}$ | $CAR^{ANN}$ | $CAR^{DRIFT}$ | $CAR^{DRIFT}$ |
| 窗口 | $[d,\,d{+}1]$ | $[d,\,d{+}1]$ | $[d{+}2,\,d{+}61]$ | $[d{+}2,\,d{+}61]$ |
| $LLM$ 口径 | first | average | first | average |
| $SUE$ | 0.0622\*\*\* | 0.0563\*\*\* | 0.0176\*\*\* | 0.0176\*\*\* |
| | (0.0019) | (0.0019) | (0.0059) | (0.0059) |
| $ATT$ | −0.0009\*\*\* | −0.0009\*\*\* | −0.0002 | −0.0003 |
| | (0.0002) | (0.0002) | (0.0006) | (0.0006) |
| $LLM$ | 9.3609\*\*\* | 14.0247\*\*\* | 1.9821 | 2.6540 |
| | (0.5397) | (0.6265) | (1.3493) | (1.8566) |
| **$ATT\times LLM$** | −0.3324\*\*\* | −0.1816\*\* | −0.3486\*\* | −0.4694\*\* |
| | (0.0662) | (0.0789) | (0.1697) | (0.2310) |
| $SUE\times ATT$ | 0.0027\*\*\* | 0.0023\*\*\* | 0.0011 | 0.0012 |
| | (0.0003) | (0.0003) | (0.0009) | (0.0010) |
| $SUE\times LLM$ | 0.9159 | 1.3722\*\* | 5.2867\*\*\* | 5.3819\*\*\* |
| | (0.5980) | (0.6314) | (1.6302) | (2.0270) |
| | | | | |
| 控制变量（10） | Yes | Yes | Yes | Yes |
| 控制变量 $\times\,SUE$ | No | No | No | No |
| 控制变量 $\times\,LLM$ | No | No | No | No |
| 年 / 月 / 星期 / 行业 FE | Yes | Yes | Yes | Yes |
| 标准误聚类 | 公告日 | 公告日 | 公告日 | 公告日 |
| 观测数 | 138,523 | 138,523 | 138,523 | 138,523 |
| Within $R^2$ | 0.1296 | 0.1530 | 0.0026 | 0.0025 |

*注：* 系数下方括号内为按公告日聚类的稳健标准误（5,069 个 cluster）。\* $p<0.10$，\*\* $p<0.05$，\*\*\* $p<0.01$。$SUE$ 为盈余意外的当季十分位缩放到 $[0,1]$；$ATT=11-NRANK$，取值 1–10；$LLM$ 为事件窗口 $[d,\,d{+}1]$ 内新闻的 LLM 预测，first 取时间戳最早一条、average 取全部新闻按条平均。控制变量 10 个：SIZE、BM、LNANALYST、LAG、LAG²、LAG³、IO、EVOL、EPERSIST、TURN。收益口径为 open-to-open。样本 2004–2025 年，4,015 家公司。

**表中读到的**

- $b_4$ 四列全为负且全部 $p<0.05$。相对设定 (1)，DRIFT 两列由 −0.3182 / −0.4151
  （$p\approx0.06$）变为 −0.3486 / −0.4694（$p=0.040$ / $0.042$）；ANN 列 average
  由 −0.0668（$p=0.39$）变为 −0.1816（$p=0.021$）。
- $b_6$ 在 DRIFT 两列为 5.2867 与 5.3819，均 $p<0.01$；在 ANN 两列为 0.9159
  （$p=0.126$）与 1.3722（$p=0.030$）。
- $b_3$ 在 DRIFT 两列由设定 (1) 的 4.4314 / 4.9897（$p<0.01$）降至 1.9821 / 2.6540
  （$p=0.142$ / $0.153$）；ANN 两列基本不变（9.3520→9.3609，14.0910→14.0247）。
- $b_5$ 在 ANN 两列为 0.0027 / 0.0023，均 $p<0.01$；DRIFT 两列 0.0011 / 0.0012，
  $p=0.24$ / $0.22$。
- $b_2$ 在加入 $b_5$ 后由设定 (1) 的正号转为负号，ANN 两列 −0.0009（$p<0.01$）。
- Within $R^2$ 相对设定 (1) 的增幅：ANN 列 +0.0009 / +0.0007，DRIFT 列 +0.0002 / +0.0001。

## 5.3 设定 (3)：HLT 完整

$$
CAR^{w}_{i,d}=\ \cdots
+\underbrace{b_4\,(ATT\times LLM)}_{\text{核心}}+b_5\,(SUE\times ATT)+b_6\,(SUE\times LLM)
+\sum_{k=1}^{10}c_k X_k
+\sum_{k=1}^{10}d_k (X_k\times SUE)+\sum_{k=1}^{10}f_k (X_k\times LLM)+\ \cdots
$$

回归元 36 个（不含吸收的固定效应）。

**表 3　HLT 完整设定**

| | (1) | (2) | (3) | (4) |
|:---|---:|---:|---:|---:|
| 被解释变量 | $CAR^{ANN}$ | $CAR^{ANN}$ | $CAR^{DRIFT}$ | $CAR^{DRIFT}$ |
| 窗口 | $[d,\,d{+}1]$ | $[d,\,d{+}1]$ | $[d{+}2,\,d{+}61]$ | $[d{+}2,\,d{+}61]$ |
| $LLM$ 口径 | first | average | first | average |
| $SUE$ | 0.0308\*\*\* | 0.0332\*\*\* | 0.1016\*\*\* | 0.0973\*\*\* |
| | (0.0112) | (0.0111) | (0.0357) | (0.0375) |
| $ATT$ | −0.0013\*\*\* | −0.0013\*\*\* | 0.0000 | −0.0000 |
| | (0.0002) | (0.0002) | (0.0007) | (0.0007) |
| $LLM$ | 5.9946\*\* | 2.7736 | 8.0649 | 13.1793 |
| | (2.3779) | (2.4491) | (5.8836) | (8.3160) |
| **$ATT\times LLM$** | −0.1539\*\* | 0.0147 | −0.2138 | −0.4130 |
| | (0.0729) | (0.0851) | (0.1895) | (0.2742) |
| $SUE\times ATT$ | 0.0031\*\*\* | 0.0027\*\*\* | 0.0002 | 0.0005 |
| | (0.0003) | (0.0003) | (0.0011) | (0.0012) |
| $SUE\times LLM$ | 0.4285 | 1.3582\*\* | 5.8960\*\*\* | 5.5864\*\*\* |
| | (0.5904) | (0.6213) | (1.6721) | (1.9273) |
| | | | | |
| 控制变量（10） | Yes | Yes | Yes | Yes |
| 控制变量 $\times\,SUE$ | Yes | Yes | Yes | Yes |
| 控制变量 $\times\,LLM$ | Yes | Yes | Yes | Yes |
| 年 / 月 / 星期 / 行业 FE | Yes | Yes | Yes | Yes |
| 标准误聚类 | 公告日 | 公告日 | 公告日 | 公告日 |
| 观测数 | 138,523 | 138,523 | 138,523 | 138,523 |
| Within $R^2$ | 0.1367 | 0.1611 | 0.0038 | 0.0038 |

*注：* 系数下方括号内为按公告日聚类的稳健标准误（5,069 个 cluster）。\* $p<0.10$，\*\* $p<0.05$，\*\*\* $p<0.01$。$SUE$ 为盈余意外的当季十分位缩放到 $[0,1]$；$ATT=11-NRANK$，取值 1–10；$LLM$ 为事件窗口 $[d,\,d{+}1]$ 内新闻的 LLM 预测，first 取时间戳最早一条、average 取全部新闻按条平均。控制变量 10 个：SIZE、BM、LNANALYST、LAG、LAG²、LAG³、IO、EVOL、EPERSIST、TURN。收益口径为 open-to-open。样本 2004–2025 年，4,015 家公司。

**表中读到的**

- $b_4$ 三列为负、一列为正（ANN average 为 +0.0147，$p=0.86$）。四列中只有 ANN first
  的 −0.1539 达到 $p<0.05$；DRIFT 两列 $p=0.259$ / $0.132$。
- $b_6$ 在 DRIFT 两列为 5.8960 与 5.5864，均 $p<0.01$，量级高于设定 (2) 的
  5.2867 / 5.3819。
- $b_3$ 的标准误较设定 (2) 放大 3.6–4.5 倍（ANN first 0.5397→2.3779，
  DRIFT average 1.8566→8.3160），四列中仅 ANN first 达到 $p<0.05$。
- $b_1$ 在 DRIFT 两列由设定 (2) 的 0.0176 升至 0.1016 / 0.0973，标准误由 0.0059
  升至 0.0357 / 0.0375。
- $f_k$ 中在四列均 $p<0.01$ 的两项：$SIZE\times LLM$ 与 $EVOL\times LLM$。
- Within $R^2$ 相对设定 (2)：ANN 列 +0.0071 / +0.0081，DRIFT 列 +0.0012 / +0.0013。

## 5.4 核心系数三档并列

**表 4　$b_4$（$ATT\times LLM$）在三档设定下**

| | (1) | (2) | (3) | (4) |
|:---|---:|---:|---:|---:|
| 窗口 | ANN | ANN | DRIFT | DRIFT |
| 口径 | first | average | first | average |
| 设定 (1) 最简 | −0.2456\*\*\* | −0.0668 | −0.3182\* | −0.4151\* |
| | (0.0663) | (0.0776) | (0.1700) | (0.2178) |
| 设定 (2) 两两交互全齐 | −0.3324\*\*\* | −0.1816\*\* | −0.3486\*\* | −0.4694\*\* |
| | (0.0662) | (0.0789) | (0.1697) | (0.2310) |
| 设定 (3) HLT 完整 | −0.1539\*\* | 0.0147 | −0.2138 | −0.4130 |
| | (0.0729) | (0.0851) | (0.1895) | (0.2742) |

**表 5　$b_6$（$SUE\times LLM$）**

| | ANN first | ANN average | DRIFT first | DRIFT average |
|:---|---:|---:|---:|---:|
| 设定 (2) | 0.9159 | 1.3722\*\* | 5.2867\*\*\* | 5.3819\*\*\* |
| | (0.5980) | (0.6314) | (1.6302) | (2.0270) |
| 设定 (3) | 0.4285 | 1.3582\*\* | 5.8960\*\*\* | 5.5864\*\*\* |
| | (0.5904) | (0.6213) | (1.6721) | (1.9273) |

**表 6　$b_5$（$SUE\times ATT$）**

| | ANN first | ANN average | DRIFT first | DRIFT average |
|:---|---:|---:|---:|---:|
| 设定 (2) | 0.0027\*\*\* | 0.0023\*\*\* | 0.0011 | 0.0012 |
| | (0.0003) | (0.0003) | (0.0009) | (0.0010) |
| 设定 (3) | 0.0031\*\*\* | 0.0027\*\*\* | 0.0002 | 0.0005 |
| | (0.0003) | (0.0003) | (0.0011) | (0.0012) |

**表中读到的**

- $b_4$ 的 12 格中 11 格为负。DRIFT 的 6 格全为负，其中 4 格 $p<0.10$、2 格 $p<0.05$。
- $b_4$ 在两种口径下同号：12 组两两比较中，ANN 设定 (3) 一组异号，其余同号。
- $b_6$ 在 DRIFT 的 4 格全部 $p\le0.008$，取值 5.29–5.90；在 ANN 的 4 格取值 0.43–1.37。
- $b_5$ 在 ANN 的 4 格全部 $p<0.01$，在 DRIFT 的 4 格全部 $p>0.20$。

## 5.5 边际效应：$LLM$ 的作用随 $ATT$ 的变化

设定 (2)、DRIFT 窗口下

$$\frac{\partial\,CAR^{DRIFT}}{\partial\,LLM}=b_3+b_4\cdot ATT+b_6\cdot SUE$$

在 $SUE$ 取样本均值 0.5072 处代入两种口径的系数：

**表 7　$LLM$ 对漂移的边际效应**

| $ATT$ | first | 折合 1 个标准差 | average | 折合 1 个标准差 |
|---:|---:|---:|---:|---:|
| 1 | 4.31 | 0.65pp | 4.91 | 0.68pp |
| 5 | 2.92 | 0.44pp | 3.04 | 0.42pp |
| 10 | 1.18 | 0.18pp | 0.69 | 0.10pp |

标准差取自 §3：first 为 0.0015106，average 为 0.0013835。
从 $ATT=1$ 到 $ATT=10$，边际效应下降 73%（first）与 86%（average）。
`marginsplot` 的置信区间：first 在 $ATT\le9$ 时不含 0，$ATT=10$ 时下界为 −0.35。

```stata
reghdfe car_drift_o2o sue_rank att llm_first ///
    c.att#c.llm_first c.sue_rank#c.att c.sue_rank#c.llm_first ///
    $controls, absorb(year month dow ff10) vce(cluster date_id)

margins, dydx(llm_first) at(att = (1(1)10))
marginsplot, yline(0) ///
    title("Marginal effect of LLM signal on drift, by attention") ///
    xtitle("ATT (higher = more attention available)") ///
    ytitle("dCAR[2,61] / dLLM")
```

## 5.6 生成论文体例表格的 Stata 代码

十二个模型先全部 `eststo` 存好，再按设定分三组导出，每组四列。
三个选项做出论文体例：`coeflabels()` 换掉 Stata 的变量名，
`indicate()` 把 10 个控制变量折成一行 Yes / No，`booktabs` 出三线表。

```stata
global controls size_dec bm_dec lnanalyst lag lag2 lag3 io evol epersist turn
global FE  absorb(year month dow ff10) vce(cluster date_id)

eststo clear

foreach s in first avg {
    reghdfe car_ann_o2o   sue_rank att llm_`s' c.att#c.llm_`s' $controls, $FE
    eststo a1_`s'
    reghdfe car_drift_o2o sue_rank att llm_`s' c.att#c.llm_`s' $controls, $FE
    eststo d1_`s'

    reghdfe car_ann_o2o   sue_rank att llm_`s' ///
        c.att#c.llm_`s' c.sue_rank#c.att c.sue_rank#c.llm_`s' $controls, $FE
    eststo a2_`s'
    reghdfe car_drift_o2o sue_rank att llm_`s' ///
        c.att#c.llm_`s' c.sue_rank#c.att c.sue_rank#c.llm_`s' $controls, $FE
    eststo d2_`s'

    reghdfe car_ann_o2o   sue_rank att llm_`s' ///
        c.att#c.llm_`s' c.sue_rank#c.att c.sue_rank#c.llm_`s' ///
        $controls c.($controls)#c.sue_rank c.($controls)#c.llm_`s', $FE
    eststo a3_`s'
    reghdfe car_drift_o2o sue_rank att llm_`s' ///
        c.att#c.llm_`s' c.sue_rank#c.att c.sue_rank#c.llm_`s' ///
        $controls c.($controls)#c.sue_rank c.($controls)#c.llm_`s', $FE
    eststo d3_`s'
}

* 把 avg 的系数改名到 first 的行上，两种口径才能并排
global RN rename(llm_avg llm_first ///
                 c.att#c.llm_avg c.att#c.llm_first ///
                 c.sue_rank#c.llm_avg c.sue_rank#c.llm_first)

* 论文体例的行标签
global LAB coeflabels(sue_rank "SUE" att "ATT" llm_first "LLM" ///
                      c.att#c.llm_first "ATT x LLM" ///
                      c.sue_rank#c.att "SUE x ATT" ///
                      c.sue_rank#c.llm_first "SUE x LLM")

global OPT b(4) se(4) star(* 0.10 ** 0.05 *** 0.01) nogaps ///
           keep(sue_rank att llm_first c.att#c.llm_first ///
                c.sue_rank#c.att c.sue_rank#c.llm_first) ///
           order(sue_rank att llm_first c.att#c.llm_first ///
                 c.sue_rank#c.att c.sue_rank#c.llm_first) ///
           indicate("Controls (10) = $controls") ///
           mgroups("CAR-ANN [d,d+1]" "CAR-DRIFT [d+2,d+61]", pattern(1 0 1 0)) ///
           mtitles("first" "average" "first" "average") ///
           stats(N r2_within, fmt(%9.0gc %9.4f) labels("Observations" "Within R2")) ///
           nonotes ///
           addnotes("Cluster-robust standard errors by announcement date in parentheses." ///
                    "Year, month, day-of-week and FF10 industry fixed effects absorbed." ///
                    "* p<0.10, ** p<0.05, *** p<0.01")

esttab a1_first a1_avg d1_first d1_avg using "build/tab1.rtf", replace $RN $LAB $OPT ///
    title("Table 1. Minimal specification")

esttab a2_first a2_avg d2_first d2_avg using "build/tab2.rtf", replace $RN $LAB $OPT ///
    title("Table 2. All pairwise interactions")

esttab a3_first a3_avg d3_first d3_avg using "build/tab3.rtf", replace $RN $LAB $OPT ///
    title("Table 3. HLT full specification")
```

出 LaTeX 三线表时把扩展名换成 `.tex` 并加 `booktabs`：

```stata
esttab a2_first a2_avg d2_first d2_avg using "build/tab2.tex", replace ///
    booktabs $RN $LAB $OPT title("Table 2. All pairwise interactions\label{tab2}")
```

附录全系数表去掉 `keep()`、`order()` 与 `indicate()` 即可：

```stata
esttab a1_first a1_avg d1_first d1_avg using "build/tabA1.rtf", replace $RN $LAB ///
    b(4) se(4) star(* 0.10 ** 0.05 *** 0.01) nogaps ///
    mgroups("CAR-ANN [d,d+1]" "CAR-DRIFT [d+2,d+61]", pattern(1 0 1 0)) ///
    mtitles("first" "average" "first" "average") ///
    stats(N r2_within, fmt(%9.0gc %9.4f) labels("Observations" "Within R2")) ///
    title("Table A1. Minimal specification, all coefficients")
```

想把括号里的标准误换成 $t$ 值，把 `se(4)` 改成 `t(2)`。

---

# 6. 附录：全系数表

正表以 Yes / No 标示的控制变量在这里逐个列出。列的编排与正表相同。

## 附表 A1　设定 (1) 全系数

| | (1) | (2) | (3) | (4) |
|:---|---:|---:|---:|---:|
| 被解释变量 | $CAR^{ANN}$ | $CAR^{ANN}$ | $CAR^{DRIFT}$ | $CAR^{DRIFT}$ |
| 窗口 | $[d,\,d{+}1]$ | $[d,\,d{+}1]$ | $[d{+}2,\,d{+}61]$ | $[d{+}2,\,d{+}61]$ |
| $LLM$ 口径 | first | average | first | average |
| $SUE$ | 0.0774\*\*\* | 0.0692\*\*\* | 0.0269\*\*\* | 0.0262\*\*\* |
| | (0.0008) | (0.0008) | (0.0025) | (0.0026) |
| $ATT$ | 0.0004\*\*\* | 0.0002 | 0.0003 | 0.0003 |
| | (0.0001) | (0.0001) | (0.0004) | (0.0004) |
| $LLM$ | 9.3520\*\*\* | 14.0910\*\*\* | 4.4314\*\*\* | 4.9897\*\*\* |
| | (0.4576) | (0.5458) | (1.1769) | (1.4748) |
| **$ATT\times LLM$** | −0.2456\*\*\* | −0.0668 | −0.3182\* | −0.4151\* |
| | (0.0663) | (0.0776) | (0.1700) | (0.2178) |
| SIZE | −0.0003\*\* | −0.0008\*\*\* | 0.0007\* | 0.0006 |
| | (0.0001) | (0.0001) | (0.0004) | (0.0004) |
| BM | −0.0003\*\*\* | −0.0002\*\* | −0.0005\* | −0.0005\*\* |
| | (0.0001) | (0.0001) | (0.0003) | (0.0003) |
| LNANALYST | −0.0011\*\* | 0.0011\*\* | 0.0011 | 0.0014 |
| | (0.0005) | (0.0005) | (0.0016) | (0.0016) |
| LAG | −0.0003 | −0.0002 | −0.0004 | −0.0005 |
| | (0.0002) | (0.0002) | (0.0005) | (0.0005) |
| LAG² | 0.0000 | 0.0000 | 0.0000\* | 0.0000\* |
| | (0.0000) | (0.0000) | (0.0000) | (0.0000) |
| LAG³ | 0.0000 | 0.0000 | −0.0000\*\* | −0.0000\*\* |
| | (0.0000) | (0.0000) | (0.0000) | (0.0000) |
| IO | −0.0016 | −0.0031\*\*\* | 0.0047 | 0.0046 |
| | (0.0010) | (0.0010) | (0.0035) | (0.0035) |
| EVOL | −0.0000\*\*\* | −0.0000\*\*\* | 0.0000 | 0.0000 |
| | (0.0000) | (0.0000) | (0.0000) | (0.0000) |
| EPERSIST | −0.0001 | −0.0007 | 0.0011 | 0.0011 |
| | (0.0006) | (0.0006) | (0.0019) | (0.0019) |
| TURN | −0.0009 | 0.0011 | 0.0036 | 0.0038 |
| | (0.0019) | (0.0019) | (0.0048) | (0.0048) |
| 常数项 | −0.0345\*\*\* | −0.0349\*\*\* | −0.0282\*\*\* | −0.0271\*\*\* |
| | (0.0030) | (0.0030) | (0.0095) | (0.0095) |
| | | | | |
| 观测数 | 138,523 | 138,523 | 138,523 | 138,523 |
| Within $R^2$ | 0.1287 | 0.1523 | 0.0024 | 0.0024 |

*注：* 系数下方括号内为按公告日聚类的稳健标准误（5,069 个 cluster）。\* $p<0.10$，\*\* $p<0.05$，\*\*\* $p<0.01$。$SUE$ 为盈余意外的当季十分位缩放到 $[0,1]$；$ATT=11-NRANK$，取值 1–10；$LLM$ 为事件窗口 $[d,\,d{+}1]$ 内新闻的 LLM 预测，first 取时间戳最早一条、average 取全部新闻按条平均。控制变量 10 个：SIZE、BM、LNANALYST、LAG、LAG²、LAG³、IO、EVOL、EPERSIST、TURN。收益口径为 open-to-open。样本 2004–2025 年，4,015 家公司。

## 附表 A2　设定 (2) 全系数

| | (1) | (2) | (3) | (4) |
|:---|---:|---:|---:|---:|
| 被解释变量 | $CAR^{ANN}$ | $CAR^{ANN}$ | $CAR^{DRIFT}$ | $CAR^{DRIFT}$ |
| 窗口 | $[d,\,d{+}1]$ | $[d,\,d{+}1]$ | $[d{+}2,\,d{+}61]$ | $[d{+}2,\,d{+}61]$ |
| $LLM$ 口径 | first | average | first | average |
| $SUE$ | 0.0622\*\*\* | 0.0563\*\*\* | 0.0176\*\*\* | 0.0176\*\*\* |
| | (0.0019) | (0.0019) | (0.0059) | (0.0059) |
| $ATT$ | −0.0009\*\*\* | −0.0009\*\*\* | −0.0002 | −0.0003 |
| | (0.0002) | (0.0002) | (0.0006) | (0.0006) |
| $LLM$ | 9.3609\*\*\* | 14.0247\*\*\* | 1.9821 | 2.6540 |
| | (0.5397) | (0.6265) | (1.3493) | (1.8566) |
| **$ATT\times LLM$** | −0.3324\*\*\* | −0.1816\*\* | −0.3486\*\* | −0.4694\*\* |
| | (0.0662) | (0.0789) | (0.1697) | (0.2310) |
| $SUE\times ATT$ | 0.0027\*\*\* | 0.0023\*\*\* | 0.0011 | 0.0012 |
| | (0.0003) | (0.0003) | (0.0009) | (0.0010) |
| $SUE\times LLM$ | 0.9159 | 1.3722\*\* | 5.2867\*\*\* | 5.3819\*\*\* |
| | (0.5980) | (0.6314) | (1.6302) | (2.0270) |
| SIZE | −0.0003\*\* | −0.0008\*\*\* | 0.0007\* | 0.0006\* |
| | (0.0001) | (0.0001) | (0.0004) | (0.0004) |
| BM | −0.0003\*\*\* | −0.0002\*\*\* | −0.0006\*\* | −0.0006\*\* |
| | (0.0001) | (0.0001) | (0.0003) | (0.0003) |
| LNANALYST | −0.0011\* | 0.0011\*\* | 0.0012 | 0.0015 |
| | (0.0005) | (0.0005) | (0.0016) | (0.0016) |
| LAG | −0.0003\* | −0.0002 | −0.0005 | −0.0005 |
| | (0.0002) | (0.0002) | (0.0005) | (0.0005) |
| LAG² | 0.0000 | 0.0000 | 0.0000\* | 0.0000\* |
| | (0.0000) | (0.0000) | (0.0000) | (0.0000) |
| LAG³ | 0.0000 | −0.0000 | −0.0000\*\* | −0.0000\*\* |
| | (0.0000) | (0.0000) | (0.0000) | (0.0000) |
| IO | −0.0016 | −0.0032\*\*\* | 0.0046 | 0.0045 |
| | (0.0010) | (0.0010) | (0.0035) | (0.0035) |
| EVOL | −0.0000\*\*\* | −0.0000\*\*\* | 0.0000 | 0.0000 |
| | (0.0000) | (0.0000) | (0.0000) | (0.0000) |
| EPERSIST | −0.0001 | −0.0006 | 0.0012 | 0.0012 |
| | (0.0006) | (0.0006) | (0.0019) | (0.0019) |
| TURN | −0.0010 | 0.0010 | 0.0035 | 0.0037 |
| | (0.0019) | (0.0019) | (0.0048) | (0.0048) |
| 常数项 | −0.0268\*\*\* | −0.0284\*\*\* | −0.0240\*\* | −0.0232\*\* |
| | (0.0031) | (0.0031) | (0.0100) | (0.0099) |
| | | | | |
| 观测数 | 138,523 | 138,523 | 138,523 | 138,523 |
| Within $R^2$ | 0.1296 | 0.1530 | 0.0026 | 0.0025 |

*注：* 系数下方括号内为按公告日聚类的稳健标准误（5,069 个 cluster）。\* $p<0.10$，\*\* $p<0.05$，\*\*\* $p<0.01$。$SUE$ 为盈余意外的当季十分位缩放到 $[0,1]$；$ATT=11-NRANK$，取值 1–10；$LLM$ 为事件窗口 $[d,\,d{+}1]$ 内新闻的 LLM 预测，first 取时间戳最早一条、average 取全部新闻按条平均。控制变量 10 个：SIZE、BM、LNANALYST、LAG、LAG²、LAG³、IO、EVOL、EPERSIST、TURN。收益口径为 open-to-open。样本 2004–2025 年，4,015 家公司。

## 附表 A3　设定 (3) 全系数

| | (1) | (2) | (3) | (4) |
|:---|---:|---:|---:|---:|
| 被解释变量 | $CAR^{ANN}$ | $CAR^{ANN}$ | $CAR^{DRIFT}$ | $CAR^{DRIFT}$ |
| 窗口 | $[d,\,d{+}1]$ | $[d,\,d{+}1]$ | $[d{+}2,\,d{+}61]$ | $[d{+}2,\,d{+}61]$ |
| $LLM$ 口径 | first | average | first | average |
| $SUE$ | 0.0308\*\*\* | 0.0332\*\*\* | 0.1016\*\*\* | 0.0973\*\*\* |
| | (0.0112) | (0.0111) | (0.0357) | (0.0375) |
| $ATT$ | −0.0013\*\*\* | −0.0013\*\*\* | 0.0000 | −0.0000 |
| | (0.0002) | (0.0002) | (0.0007) | (0.0007) |
| $LLM$ | 5.9946\*\* | 2.7736 | 8.0649 | 13.1793 |
| | (2.3779) | (2.4491) | (5.8836) | (8.3160) |
| **$ATT\times LLM$** | −0.1539\*\* | 0.0147 | −0.2138 | −0.4130 |
| | (0.0729) | (0.0851) | (0.1895) | (0.2742) |
| $SUE\times ATT$ | 0.0031\*\*\* | 0.0027\*\*\* | 0.0002 | 0.0005 |
| | (0.0003) | (0.0003) | (0.0011) | (0.0012) |
| $SUE\times LLM$ | 0.4285 | 1.3582\*\* | 5.8960\*\*\* | 5.5864\*\*\* |
| | (0.5904) | (0.6213) | (1.6721) | (1.9273) |
| | | | | |
| *控制变量主效应 $c_k$* | | | | |
| SIZE | 0.0023\*\*\* | 0.0017\*\*\* | 0.0013\* | 0.0012 |
| | (0.0002) | (0.0002) | (0.0007) | (0.0007) |
| BM | 0.0013\*\*\* | 0.0012\*\*\* | −0.0012\*\* | −0.0013\*\* |
| | (0.0002) | (0.0002) | (0.0005) | (0.0005) |
| LNANALYST | −0.0033\*\*\* | −0.0010 | 0.0075\*\* | 0.0076\*\*\* |
| | (0.0010) | (0.0010) | (0.0030) | (0.0029) |
| LAG | −0.0011\*\*\* | −0.0010\*\*\* | 0.0003 | 0.0002 |
| | (0.0003) | (0.0003) | (0.0010) | (0.0010) |
| LAG² | 0.0000\*\*\* | 0.0000\*\*\* | 0.0000 | 0.0000 |
| | (0.0000) | (0.0000) | (0.0000) | (0.0000) |
| LAG³ | −0.0000\*\*\* | −0.0000\*\*\* | −0.0000 | −0.0000 |
| | (0.0000) | (0.0000) | (0.0000) | (0.0000) |
| IO | −0.0309\*\*\* | −0.0311\*\*\* | 0.0278\*\*\* | 0.0268\*\*\* |
| | (0.0018) | (0.0018) | (0.0062) | (0.0060) |
| EVOL | −0.0000\*\*\* | −0.0000\*\*\* | 0.0000\* | 0.0000 |
| | (0.0000) | (0.0000) | (0.0000) | (0.0000) |
| EPERSIST | 0.0013 | 0.0008 | −0.0036 | −0.0036 |
| | (0.0013) | (0.0013) | (0.0044) | (0.0043) |
| TURN | −0.0054\*\*\* | −0.0008 | 0.0029 | −0.0002 |
| | (0.0017) | (0.0017) | (0.0058) | (0.0070) |
| | | | | |
| *控制变量 $\times\,SUE$，$d_k$* | | | | |
| SIZE × SUE | −0.0031\*\*\* | −0.0033\*\*\* | −0.0009 | −0.0012 |
| | (0.0004) | (0.0004) | (0.0012) | (0.0013) |
| BM × SUE | −0.0026\*\*\* | −0.0021\*\*\* | 0.0020\*\* | 0.0021\*\* |
| | (0.0003) | (0.0003) | (0.0010) | (0.0010) |
| LNANALYST × SUE | 0.0007 | −0.0025 | −0.0108\*\* | −0.0111\*\* |
| | (0.0018) | (0.0018) | (0.0050) | (0.0051) |
| LAG × SUE | 0.0013\* | 0.0012\* | −0.0013 | −0.0011 |
| | (0.0007) | (0.0006) | (0.0018) | (0.0019) |
| LAG² × SUE | −0.0000\*\* | −0.0000\*\* | 0.0000 | −0.0000 |
| | (0.0000) | (0.0000) | (0.0000) | (0.0000) |
| LAG³ × SUE | 0.0000\* | 0.0000\* | 0.0000 | 0.0000 |
| | (0.0000) | (0.0000) | (0.0000) | (0.0000) |
| IO × SUE | 0.0558\*\*\* | 0.0516\*\*\* | −0.0408\*\*\* | −0.0381\*\*\* |
| | (0.0033) | (0.0033) | (0.0110) | (0.0119) |
| EVOL × SUE | 0.0000\*\*\* | 0.0000\*\*\* | −0.0000\* | −0.0000 |
| | (0.0000) | (0.0000) | (0.0000) | (0.0000) |
| EPERSIST × SUE | −0.0006 | −0.0006 | 0.0101 | 0.0099 |
| | (0.0024) | (0.0024) | (0.0079) | (0.0080) |
| TURN × SUE | 0.0079\*\* | 0.0056 | 0.0032 | 0.0062 |
| | (0.0035) | (0.0041) | (0.0106) | (0.0116) |
| | | | | |
| *控制变量 $\times\,LLM$，$f_k$* | | | | |
| SIZE × LLM | −1.1373\*\*\* | −1.1421\*\*\* | −0.1327 | −0.1341 |
| | (0.0940) | (0.1002) | (0.2323) | (0.2801) |
| BM × LLM | −0.3060\*\*\* | −0.5518\*\*\* | −0.3576\*\* | −0.5228\*\* |
| | (0.0623) | (0.0660) | (0.1623) | (0.2188) |
| LNANALYST × LLM | 2.3376\*\*\* | 5.3478\*\*\* | −1.1878 | 0.0683 |
| | (0.3816) | (0.4144) | (0.9732) | (1.1468) |
| LAG × LLM | 0.2041 | 0.2431\* | −0.0639 | −0.3233 |
| | (0.1395) | (0.1362) | (0.3178) | (0.4015) |
| LAG² × LLM | −0.0026 | −0.0030 | 0.0031 | 0.0081 |
| | (0.0029) | (0.0026) | (0.0061) | (0.0075) |
| LAG³ × LLM | 0.0000 | 0.0000 | −0.0000 | −0.0000 |
| | (0.0000) | (0.0000) | (0.0000) | (0.0000) |
| IO × LLM | 1.5621\*\* | 1.5168\* | −4.6046\*\* | −6.0716\*\* |
| | (0.7571) | (0.8354) | (2.2676) | (2.7698) |
| EVOL × LLM | 0.0005\*\*\* | 0.0006\*\*\* | −0.0018\*\*\* | −0.0019\*\*\* |
| | (0.0001) | (0.0002) | (0.0004) | (0.0005) |
| EPERSIST × LLM | −0.8987\*\* | −1.3850\*\*\* | −0.3152 | −0.2556 |
| | (0.4553) | (0.4877) | (1.2626) | (1.4426) |
| TURN × LLM | −1.8032 | 2.1599\* | 1.7396 | −2.9934 |
| | (1.1093) | (1.2496) | (2.9279) | (3.6891) |
| 常数项 | −0.0089 | −0.0094\* | −0.0703\*\*\* | −0.0686\*\*\* |
| | (0.0058) | (0.0057) | (0.0197) | (0.0192) |
| | | | | |
| 观测数 | 138,523 | 138,523 | 138,523 | 138,523 |
| Within $R^2$ | 0.1367 | 0.1611 | 0.0038 | 0.0038 |

*注：* 系数下方括号内为按公告日聚类的稳健标准误（5,069 个 cluster）。\* $p<0.10$，\*\* $p<0.05$，\*\*\* $p<0.01$。$SUE$ 为盈余意外的当季十分位缩放到 $[0,1]$；$ATT=11-NRANK$，取值 1–10；$LLM$ 为事件窗口 $[d,\,d{+}1]$ 内新闻的 LLM 预测，first 取时间戳最早一条、average 取全部新闻按条平均。控制变量 10 个：SIZE、BM、LNANALYST、LAG、LAG²、LAG³、IO、EVOL、EPERSIST、TURN。收益口径为 open-to-open。样本 2004–2025 年，4,015 家公司。